In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, 
                           recall_score, f1_score, confusion_matrix)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def machine_learning_algorithm_selection():
    """
    Seleção de algoritmos de machine learning para ambos os modelos
    usando nested cross-validation e múltiplos algoritmos
    """
    
    print("="*100)
    print("MACHINE LEARNING ALGORITHM SELECTION - MODELOS 1 E 2")
    print("="*100)
    
    # Carregar dados de ambos os modelos
    model1_train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TRAIN.csv'
    model2_train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TRAIN.csv'
    
    df_model1 = pd.read_csv(model1_train_path)
    df_model2 = pd.read_csv(model2_train_path)
    
    # Corrigir tipos de dados do Modelo 2
    binary_vars_model2 = [
        'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
        'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
        'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
        'usou_utensilios_amamentacao'
    ]
    
    for var in binary_vars_model2:
        if var in df_model2.columns:
            df_model2[var] = df_model2[var].astype(int)
    
    # Preparar dados
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    
    # Modelo 1
    y_model1 = df_model1['obeso'].copy()
    X_model1 = df_model1.drop(target_cols + ['id_anon'], axis=1)
    
    # Modelo 2  
    y_model2 = df_model2['obeso'].copy()
    X_model2 = df_model2.drop(target_cols + ['id_anon'], axis=1)
    
    print(f"MODELO 1 - Fatores Demográficos/Perinatais:")
    print(f"  - Observações: {len(df_model1):,}")
    print(f"  - Features: {X_model1.shape[1]}")
    print(f"  - Obesos: {y_model1.sum()} ({y_model1.mean()*100:.1f}%)")
    
    print(f"\nMODELO 2 - Práticas de Alimentação:")
    print(f"  - Observações: {len(df_model2):,}")
    print(f"  - Features: {X_model2.shape[1]}")
    print(f"  - Obesos: {y_model2.sum()} ({y_model2.mean()*100:.1f}%)")
    
    # Configurações de cross-validation
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Definir algoritmos e hiperparâmetros
    algorithms = {
        'Logistic Regression': {
            'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0, 100.0],
                'model__penalty': ['l1', 'l2'],
                'model__solver': ['liblinear']
            }
        },
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'model__n_estimators': [50, 100, 200],
                'model__max_depth': [3, 5, 10, None],
                'model__min_samples_split': [2, 5, 10]
            }
        },
        'Decision Tree': {
            'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'model__max_depth': [3, 5, 10, 15, None],
                'model__min_samples_split': [2, 5, 10, 20],
                'model__min_samples_leaf': [1, 2, 5, 10]
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'model__n_estimators': [50, 100, 200],
                'model__learning_rate': [0.01, 0.1, 0.2],
                'model__max_depth': [3, 5, 7]
            }
        },
        'k-Nearest Neighbors': {
            'model': KNeighborsClassifier(),
            'params': {
                'model__n_neighbors': [3, 5, 7, 9, 11],
                'model__weights': ['uniform', 'distance'],
                'model__metric': ['euclidean', 'manhattan']
            }
        },
        'Gaussian Naive Bayes': {
            'model': GaussianNB(),
            'params': {
                'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
            }
        },
        'SVM (RBF)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0],
                'model__gamma': ['scale', 'auto', 0.001, 0.01],
                'model__kernel': ['rbf']
            }
        },
        'SVM (Linear)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'model__C': [0.1, 1.0, 10.0, 100.0],
                'model__kernel': ['linear']
            }
        }
    }
    
    def calculate_confidence_intervals(scores, confidence=0.95):
        """Calcula intervalos de confiança usando distribuição t"""
        n = len(scores)
        if n <= 1:
            return np.mean(scores), np.mean(scores), np.mean(scores)
        mean_score = np.mean(scores)
        std_err = stats.sem(scores)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean_score, mean_score - h, mean_score + h
    
    def evaluate_algorithm(X_data, y_data, algorithm_name, algorithm_config, model_name):
        """Avalia um algoritmo usando nested cross-validation"""
        
        print(f"\n🔍 Avaliando {algorithm_name} ({model_name})...")
        
        # Pipeline com preprocessamento
        pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('model', algorithm_config['model'])
        ])
        
        # Nested cross-validation
        auc_scores = []
        precision_scores = []
        recall_scores = []
        f1_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_data, y_data)):
            X_train_fold, X_val_fold = X_data.iloc[train_idx], X_data.iloc[val_idx]
            y_train_fold, y_val_fold = y_data.iloc[train_idx], y_data.iloc[val_idx]
            
            # Grid search no inner CV
            grid_search = GridSearchCV(
                pipeline, algorithm_config['params'], cv=inner_cv,
                scoring='roc_auc', n_jobs=-1, verbose=0
            )
            
            try:
                grid_search.fit(X_train_fold, y_train_fold)
                
                # Predições
                y_pred_proba = grid_search.predict_proba(X_val_fold)[:, 1]
                y_pred = grid_search.predict(X_val_fold)
                
                # Métricas
                auc_scores.append(roc_auc_score(y_val_fold, y_pred_proba))
                precision_scores.append(precision_score(y_val_fold, y_pred, zero_division=0))
                recall_scores.append(recall_score(y_val_fold, y_pred, zero_division=0))
                f1_scores.append(f1_score(y_val_fold, y_pred, zero_division=0))
                
            except Exception as e:
                print(f"    Erro no fold {fold}: {str(e)}")
                # Adicionar scores ruins em caso de erro
                auc_scores.append(0.5)
                precision_scores.append(0.0)
                recall_scores.append(0.0)
                f1_scores.append(0.0)
        
        # Calcular intervalos de confiança
        auc_mean, auc_ci_lower, auc_ci_upper = calculate_confidence_intervals(auc_scores)
        prec_mean, prec_ci_lower, prec_ci_upper = calculate_confidence_intervals(precision_scores)
        rec_mean, rec_ci_lower, rec_ci_upper = calculate_confidence_intervals(recall_scores)
        f1_mean, f1_ci_lower, f1_ci_upper = calculate_confidence_intervals(f1_scores)
        
        return {
            'algorithm': algorithm_name,
            'model': model_name,
            'auc': {'mean': auc_mean, 'ci_lower': auc_ci_lower, 'ci_upper': auc_ci_upper, 'std': np.std(auc_scores)},
            'precision': {'mean': prec_mean, 'ci_lower': prec_ci_lower, 'ci_upper': prec_ci_upper, 'std': np.std(precision_scores)},
            'recall': {'mean': rec_mean, 'ci_lower': rec_ci_lower, 'ci_upper': rec_ci_upper, 'std': np.std(recall_scores)},
            'f1': {'mean': f1_mean, 'ci_lower': f1_ci_lower, 'ci_upper': f1_ci_upper, 'std': np.std(f1_scores)},
            'scores': {'auc': auc_scores, 'precision': precision_scores, 'recall': recall_scores, 'f1': f1_scores}
        }
    
    # =======================================================================
    # AVALIAÇÃO DOS ALGORITMOS
    # =======================================================================
    
    all_results = []
    
    print(f"\n" + "="*80)
    print("AVALIAÇÃO DE ALGORITMOS - MODELO 1")
    print("="*80)
    
    model1_results = []
    for alg_name, alg_config in algorithms.items():
        result = evaluate_algorithm(X_model1, y_model1, alg_name, alg_config, 'Modelo 1')
        model1_results.append(result)
        all_results.append(result)
    
    print(f"\n" + "="*80)
    print("AVALIAÇÃO DE ALGORITMOS - MODELO 2")
    print("="*80)
    
    model2_results = []
    for alg_name, alg_config in algorithms.items():
        result = evaluate_algorithm(X_model2, y_model2, alg_name, alg_config, 'Modelo 2')
        model2_results.append(result)
        all_results.append(result)
    
    # =======================================================================
    # RESULTADOS COMPARATIVOS
    # =======================================================================
    
    def print_results_table(results, title):
        print(f"\n{title}:")
        print(f"{'Algoritmo':<20} {'AUC-ROC':<25} {'Precision':<25} {'Recall':<25} {'F1-Score':<25}")
        print("-" * 120)
        
        for result in results:
            alg_name = result['algorithm']
            auc = result['auc']
            precision = result['precision']
            recall = result['recall']
            f1 = result['f1']
            
            auc_str = f"{auc['mean']:.3f} ({auc['ci_lower']:.3f}-{auc['ci_upper']:.3f})"
            prec_str = f"{precision['mean']:.3f} ({precision['ci_lower']:.3f}-{precision['ci_upper']:.3f})"
            rec_str = f"{recall['mean']:.3f} ({recall['ci_lower']:.3f}-{recall['ci_upper']:.3f})"
            f1_str = f"{f1['mean']:.3f} ({f1['ci_lower']:.3f}-{f1['ci_upper']:.3f})"
            
            print(f"{alg_name:<20} {auc_str:<25} {prec_str:<25} {rec_str:<25} {f1_str:<25}")
    
    print(f"\n" + "="*100)
    print("RESULTADOS COMPARATIVOS")
    print("="*100)
    
    print_results_table(model1_results, "MODELO 1 - Fatores Demográficos/Perinatais")
    print_results_table(model2_results, "MODELO 2 - Práticas de Alimentação")
    
    # Identificar melhores algoritmos
    print(f"\n" + "="*80)
    print("MELHORES ALGORITMOS POR MODELO")
    print("="*80)
    
    # Modelo 1
    model1_sorted = sorted(model1_results, key=lambda x: x['auc']['mean'], reverse=True)
    best_model1 = model1_sorted[0]
    
    print(f"MODELO 1:")
    print(f"  Melhor algoritmo: {best_model1['algorithm']}")
    print(f"  AUC-ROC: {best_model1['auc']['mean']:.3f} (95% CI: {best_model1['auc']['ci_lower']:.3f}-{best_model1['auc']['ci_upper']:.3f})")
    print(f"  Precision: {best_model1['precision']['mean']:.3f}")
    print(f"  Recall: {best_model1['recall']['mean']:.3f}")
    print(f"  F1-Score: {best_model1['f1']['mean']:.3f}")
    
    # Modelo 2
    model2_sorted = sorted(model2_results, key=lambda x: x['auc']['mean'], reverse=True)
    best_model2 = model2_sorted[0]
    
    print(f"\nMODELO 2:")
    print(f"  Melhor algoritmo: {best_model2['algorithm']}")
    print(f"  AUC-ROC: {best_model2['auc']['mean']:.3f} (95% CI: {best_model2['auc']['ci_lower']:.3f}-{best_model2['auc']['ci_upper']:.3f})")
    print(f"  Precision: {best_model2['precision']['mean']:.3f}")
    print(f"  Recall: {best_model2['recall']['mean']:.3f}")
    print(f"  F1-Score: {best_model2['f1']['mean']:.3f}")
    
    # Comparação geral
    print(f"\n" + "="*80)
    print("ANÁLISE COMPARATIVA GERAL")
    print("="*80)
    
    if best_model2['auc']['mean'] > best_model1['auc']['mean']:
        diff = best_model2['auc']['mean'] - best_model1['auc']['mean']
        print(f"✓ Modelo 2 apresenta melhor performance (+{diff:.3f} AUC-ROC)")
    else:
        diff = best_model1['auc']['mean'] - best_model2['auc']['mean']
        print(f"✓ Modelo 1 apresenta melhor performance (+{diff:.3f} AUC-ROC)")
    
    print(f"\nLimitações observadas:")
    print(f"  - Ambos os modelos apresentam baixa precision (~3-4%)")
    print(f"  - AUC-ROC próximo a 0.6 indica capacidade preditiva limitada")
    print(f"  - Alto número de falsos positivos em ambos os casos")
    
    # Ranking geral
    print(f"\n" + "="*80)
    print("RANKING GERAL DE ALGORITMOS")
    print("="*80)
    
    all_sorted = sorted(all_results, key=lambda x: x['auc']['mean'], reverse=True)
    
    print(f"{'Posição':<5} {'Algoritmo':<20} {'Modelo':<10} {'AUC-ROC':<15} {'95% CI':<20}")
    print("-" * 75)
    
    for i, result in enumerate(all_sorted, 1):
        ci_str = f"({result['auc']['ci_lower']:.3f}-{result['auc']['ci_upper']:.3f})"
        print(f"{i:<5} {result['algorithm']:<20} {result['model']:<10} {result['auc']['mean']:<15.3f} {ci_str:<20}")
    
    return {
        'model1_results': model1_results,
        'model2_results': model2_results,
        'best_model1': best_model1,
        'best_model2': best_model2,
        'all_results': all_results
    }

# Executar seleção de algoritmos
if __name__ == "__main__":
    results = machine_learning_algorithm_selection()

MACHINE LEARNING ALGORITHM SELECTION - MODELOS 1 E 2
MODELO 1 - Fatores Demográficos/Perinatais:
  - Observações: 6,588
  - Features: 27
  - Obesos: 183 (2.8%)

MODELO 2 - Práticas de Alimentação:
  - Observações: 4,510
  - Features: 17
  - Obesos: 134 (3.0%)

AVALIAÇÃO DE ALGORITMOS - MODELO 1

🔍 Avaliando Logistic Regression (Modelo 1)...

🔍 Avaliando Random Forest (Modelo 1)...

🔍 Avaliando Decision Tree (Modelo 1)...

🔍 Avaliando Gradient Boosting (Modelo 1)...

🔍 Avaliando k-Nearest Neighbors (Modelo 1)...

🔍 Avaliando Gaussian Naive Bayes (Modelo 1)...

🔍 Avaliando SVM (RBF) (Modelo 1)...

🔍 Avaliando SVM (Linear) (Modelo 1)...

AVALIAÇÃO DE ALGORITMOS - MODELO 2

🔍 Avaliando Logistic Regression (Modelo 2)...

🔍 Avaliando Random Forest (Modelo 2)...

🔍 Avaliando Decision Tree (Modelo 2)...

🔍 Avaliando Gradient Boosting (Modelo 2)...

🔍 Avaliando k-Nearest Neighbors (Modelo 2)...

🔍 Avaliando Gaussian Naive Bayes (Modelo 2)...

🔍 Avaliando SVM (RBF) (Modelo 2)...

🔍 Avaliando SVM